
# Chapter 1: Arithmetic Functions and Multiplicatives

Arithmetic functions such as the divisor function, Euler’s totient, and the Möbius function form the backbone of multiplicative number theory.  In this chapter we follow the recipe from the book outline【155477964815303†L35-L41】—identify active mechanisms, implement them in SageMath, and explain the conceptual picture—for the first entry in the chapter’s table: the divisor function $d(n)$ (OEIS A000005)【155477964815303†L45-L53】.



## Sequence Analyst – Mechanisms & DAG for $d(n)$

**Active mechanisms**

- **ZETA (Primary):** The Dirichlet series of the divisor function is $\zeta(s)^2=\sum_{n\ge1}rac{d(n)}{n^s}$【975973847736663†L283-L289】.  Euler products show that $d(n)$ is multiplicative and encode the prime factorization of $n$.
- **MOD (Emergent):** Expanding the two‐variable generating function $\sum_{n\ge1}d(n)q^n$ gives a $q$‐series $\sum_{k\ge1}rac{q^k}{1-q^k}$ that hints at modular forms.

**Base–Bridge–Emergence DAG**

- **Base:** The combinatorial definition of $d(n)$ counts the number of divisors of $n$.  For $n=p_1^{e_1}\cdots p_r^{e_r}$ it equals $(e_1+1)\cdots(e_r+1)$.
- **Bridge:** Viewing $d(n)$ as the Dirichlet convolution of the constant function $1$ with itself ($1*1=d$) exposes its multiplicativity.  The corresponding Dirichlet series factorizes as $\zeta(s)^2$【975973847736663†L283-L289】.
- **Emergence:** Summing $d(n)q^n$ across all $n$ yields $\sum_{k\ge1}rac{q^k}{1-q^k}$, a $q$‐series reminiscent of Eisenstein series and modular forms.  This emergent mechanism connects arithmetic functions to analysis on the upper half‐plane.


In [ ]:

print("="*70)
print("DIVISOR FUNCTION d(n)")
print("="*70)


In [ ]:

# --- 1. Direct Computation ---
print("
1. First 20 values of d(n):")
# Use the divisor function sigma(n,0) which counts the number of divisors
vals = [sigma(n,0) for n in range(1,21)]
print(vals)


In [ ]:

# --- 2. Multiplicativity Check ---
print("
2. Multiplicativity: d(ab) = d(a)d(b) when gcd(a,b)=1")
# Test random coprime pairs
import random
from math import gcd

ok = True
for _ in range(10):
    a = random.randint(1,50)
    b = random.randint(1,50)
    if gcd(a,b) == 1:
        lhs = sigma(a*b,0)
        rhs = sigma(a,0) * sigma(b,0)
        if lhs != rhs:
            ok = False
            print(f"   Failure at a={a}, b={b}: d(ab)={lhs}, d(a)d(b)={rhs}")
print(f"   Multiplicativity holds? {ok}")


In [ ]:

# --- 3. Dirichlet Series: ζ(s)^2 versus Σ d(n)/n^s ---
print("
3. Dirichlet series approximation for s=2:")

# Choose s and partial sum cutoff N
s = 2
N = 200

# Compute partial sum of d(n)/n^s
partial_sum = sum(sigma(n,0) / n^s for n in range(1, N+1))

# Compute zeta(s)^2 using zeta function
zeta_val = zeta(s)
product_val = zeta_val^2

print(f"   Partial sum Σ_{1≤n≤{N}} d(n)/n^{s} ≈ {partial_sum.n(digits=12)}")
print(f"   ζ({s})^2 = {product_val.n(digits=12)}")
print(f"   Approximation error: {(product_val - partial_sum).abs().n(digits=12)}")


In [ ]:

# --- 4. Prime Power Property ---
print("
4. d(p^k) = k+1 for prime p and exponent k")

primes_list = [2,3,5,7,11]
exponents = [1,2,3,4]
all_good = True
for p in primes_list:
    for k in exponents:
        n = p^k
        lhs = sigma(n,0)
        rhs = k + 1
        if lhs != rhs:
            all_good = False
            print(f"   Mismatch at p={p}, k={k}: d(p^k)={lhs}, expected {rhs}")
print(f"   Property holds for tested primes/exponents? {all_good}")


In [ ]:

# --- 5. Dirichlet Convolution: d = 1 * 1 ---
print("
5. Dirichlet convolution: verify d(n) = (1*1)(n)")

# Define constant function 1 on positive integers
def f_const_one(n):
    return 1

# Dirichlet convolution of f and g evaluated at n

def dirichlet_convolution(f, g, n):
    return sum(f(d) * g(n//d) for d in divisors(n))

# Check for n up to 20
valid = True
for n in range(1, 21):
    conv = dirichlet_convolution(f_const_one, f_const_one, n)
    if conv != sigma(n,0):
        valid = False
        print(f"   Failure at n={n}: convolution {conv}, d(n)={sigma(n,0)}")
print(f"   Dirichlet convolution identity holds for n≤20? {valid}")


In [ ]:

# --- 6. q-Series Generating Function ---
print("
6. q-Series: Σ d(n) q^n = Σ_{k≥1} q^k/(1 - q^k)")

# Power series ring to expand both sides
R.<q> = PowerSeriesRing(QQ, default_prec=15)

# Left-hand side: sum d(n) q^n up to q^14
lhs_series = sum(sigma(n,0)*q^n for n in range(1,15))

# Right-hand side: sum over k of q^k/(1 - q^k)
rhs_series = sum(q^k/(1 - q^k) for k in range(1,8))  # finite sum approximation

print(f"   LHS coefficients: {lhs_series.list()}")
print(f"   RHS coefficients (approx): {rhs_series.list()}")

# Check first few coefficients match
def lists_match(a,b,cut):
    return all(a[i] == b[i] for i in range(cut))

print(f"   Do the first 10 coefficients agree? {lists_match(lhs_series.list(), rhs_series.list(), 10)}")



## Mathematical Expositor – Conceptual Explanations

1. **Direct computation:** Evaluating $d(n)$ for the first few $n$ reveals how the function behaves on prime powers and products.  Composite numbers with many small prime factors have larger $d(n)$.

2. **Multiplicativity:** The test confirms that $d$ is multiplicative: if $a$ and $b$ are coprime then $d(ab)=d(a)d(b)$.  This comes from the factorization of divisors over coprime components, and it is the cornerstone of multiplicative number theory.

3. **Dirichlet series:** Numerically comparing $\sum d(n)n^{-s}$ to $\zeta(s)^2$ for $s=2$ shows rapid convergence, confirming the identity $\zeta(s)^2=\sum_{n\ge1}d(n)n^{-s}$【975973847736663†L283-L289】.  The product expansion encodes the prime‐factor multiplicative structure.

4. **Prime power property:** For a prime $p$ and exponent $k$, $d(p^k)=k+1$.  This follows because the divisors are exactly $p^0, p^1,\dots,p^k$, giving $k+1$ distinct divisors.  When $n$ factorizes into distinct primes with exponents $e_i$, $d(n)=(e_1+1)\cdots(e_r+1)$.

5. **Dirichlet convolution:** The identity $d=1st 1$ expresses $d$ as the convolution of two simple constant functions.  Convolutions correspond to products of Dirichlet series; thus $D_{1}(s)^2=\zeta(s)^2$ matches the Dirichlet series of $d$.

6. **q-Series generating function:** The generating function $\sum_{n\ge1}d(n)q^n$ can be written as $\sum_{k\ge1}rac{q^k}{1-q^k}$ by grouping divisors according to their contribution.  Expanding both sides as formal power series shows that coefficients coincide for the first several terms, hinting at connections with modular forms.

These demonstrations illustrate how a simple arithmetic function links zeta functions, Dirichlet convolutions and $q$‐series.  The mechanisms identified in the DAG come to life through explicit computation.
